# 03 — Feature Engineering

**Input:**
- `df_session` — session-level CSV from `01_load_dataset.ipynb`
- `df_attempt` — attempt-level CSV from `01_load_dataset.ipynb`
- Quality gate must have passed in `02_data_quality_check.ipynb`

**Process:**
1. Aggregate attempt behavior per student × task
2. Create behavior features
3. Create task / context features
4. Create `at_risk` label
5. Exclude leakage columns
6. Group split by learner identity (`GroupShuffleSplit`)

**Output:**
- `notebooks/data/processed/X_baseline_train.parquet`
- `notebooks/data/processed/X_baseline_test.parquet`
- `notebooks/data/processed/y_train.csv`
- `notebooks/data/processed/y_test.csv`
- `notebooks/data/processed/split_metadata.json`

## Imports

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

SNAPSHOT_DATE = "YYYY-MM-DD"
BATCH_CODE    = "BATCH_XXX"

SESSION_FILE  = f"notebooks/data/raw/session_{SNAPSHOT_DATE}_{BATCH_CODE}.csv"
ATTEMPT_FILE  = f"notebooks/data/raw/attempt_{SNAPSHOT_DATE}_{BATCH_CODE}.csv"
# Phase 5 M5.3: block event CSV (None = skip block features; backward-compat with Phase 4)
EVENT_FILE    = f"notebooks/data/raw/event_{SNAPSHOT_DATE}_{BATCH_CODE}.csv"

PROCESSED_DIR = Path("notebooks/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PASS_THRESHOLD_RATIO = 0.6
TEST_SIZE            = 0.2
RANDOM_STATE         = 42

## Load datasets

In [ ]:
df_session = pd.read_csv(SESSION_FILE)
df_attempt = pd.read_csv(ATTEMPT_FILE)

# Phase 5 M5.3: load block event CSV if available
_event_path = Path(EVENT_FILE)
if _event_path.exists():
    df_event = pd.read_csv(_event_path)
    print(f"Block event rows : {len(df_event):,}")
else:
    df_event = None
    print(f"Block event CSV not found ({EVENT_FILE}) — block features will be zero-filled (Phase 4 mode)")

print(f"Session rows : {len(df_session):,}")
print(f"Attempt rows : {len(df_attempt):,}")

## Step 1 — Aggregate attempt behavior per student × task

In [ ]:
attempt_agg = (
    df_attempt
    .groupby(["academy_member_id", "task_id"])
    .agg(
        attempt_count          = ("attempt_no",        "count"),
        first_attempt_correct  = ("is_correct",        lambda x: int(x.iloc[0]) if len(x) > 0 else 0),
        attempt_correct_ratio  = ("is_correct",        "mean"),
        error_type_diversity   = ("error_type",        lambda x: x.dropna().nunique()),
        avg_execution_time_ms  = ("execution_time_ms", "mean"),
    )
    .reset_index()
)

print(f"Aggregated attempt rows : {len(attempt_agg):,}")
attempt_agg.head(3)

## Step 1b — Aggregate block events per student × task (Phase 5 M5.3)

Six block-specific features derived from `trn_event_logs` block event rows:

| Feature | Formula | Signal |
|---|---|---|
| `block_total_ops` | add + delete + move | total interaction effort |
| `block_add_count` | count of block_add | exploration depth |
| `block_delete_count` | count of block_delete | confusion / revision |
| `block_move_count` | count of block_move | reordering effort |
| `block_revision_rate` | delete / (add + 1e-6) | confusion ratio [0, ∞) |
| `block_journey_duration_sec` | max(duration_from_start) − min(duration_from_start) | active assembly time |

Non-sql_block sessions receive **zero** for all block features.
These features are leakage-safe: they are computed from interaction behavior, not from scores.

In [ ]:
BLOCK_FEATURE_COLS = [
    "block_total_ops",
    "block_add_count",
    "block_delete_count",
    "block_move_count",
    "block_revision_rate",
    "block_journey_duration_sec",
]

if df_event is not None:
    _be = df_event[
        df_event["event_type"].isin(["block_add", "block_delete", "block_move"])
    ].copy()
    _block_raw = (
        _be.groupby(["academy_member_id", "task_id"])
        .agg(
            block_add_count    = ("event_type", lambda x: (x == "block_add").sum()),
            block_delete_count = ("event_type", lambda x: (x == "block_delete").sum()),
            block_move_count   = ("event_type", lambda x: (x == "block_move").sum()),
            _dur_max           = ("duration_from_start", "max"),
            _dur_min           = ("duration_from_start", "min"),
        )
        .reset_index()
    )
    _block_raw["block_total_ops"] = (
        _block_raw["block_add_count"]
        + _block_raw["block_delete_count"]
        + _block_raw["block_move_count"]
    )
    _block_raw["block_revision_rate"] = (
        _block_raw["block_delete_count"] / (_block_raw["block_add_count"] + 1e-6)
    )
    _block_raw["block_journey_duration_sec"] = (
        _block_raw["_dur_max"] - _block_raw["_dur_min"]
    ).clip(lower=0)
    block_agg = _block_raw.drop(columns=["_dur_max", "_dur_min"])
    print(f"Block event aggregation: {len(block_agg):,} student×task pairs")
    print(block_agg[BLOCK_FEATURE_COLS].describe().T[["mean", "min", "max"]].round(2).to_string())
else:
    block_agg = pd.DataFrame(columns=["academy_member_id", "task_id"] + BLOCK_FEATURE_COLS)
    print("No event CSV — block_agg is empty (block features will be zero-filled)")

## Step 2 — Merge and create behavior features

In [ ]:
df = df_session.merge(
    attempt_agg,
    on=["academy_member_id", "task_id"],
    how="left",
)

# Fill attempt aggregates for sessions with no attempt records
df["attempt_count"]         = df["attempt_count"].fillna(0).astype(int)
df["first_attempt_correct"] = df["first_attempt_correct"].fillna(0).astype(int)
df["attempt_correct_ratio"] = df["attempt_correct_ratio"].fillna(0.0)
df["error_type_diversity"]  = df["error_type_diversity"].fillna(0).astype(int)
df["avg_execution_time_ms"] = df["avg_execution_time_ms"].fillna(0.0)

# Phase 5 M5.3: merge block features (zero-fill for non-sql_block sessions)
if len(block_agg) > 0:
    df = df.merge(block_agg, on=["academy_member_id", "task_id"], how="left")
else:
    for col in BLOCK_FEATURE_COLS:
        df[col] = 0.0
for col in BLOCK_FEATURE_COLS:
    df[col] = df[col].fillna(0.0)

print(f"Merged rows : {len(df):,}")
block_sessions = int((df["block_total_ops"] > 0).sum())
print(f"Sessions with block events : {block_sessions:,} / {len(df):,}")
df[["academy_member_id", "task_id", "total_run_count", "attempt_count",
    "first_attempt_correct", "attempt_correct_ratio", "error_type_diversity"]].head(3)

## Step 3 — Create task / context features

In [ ]:
# task_type: sql_block → 0, sql_text → 1
task_type_map = {"sql_block": 0, "sql_text": 1}
df["task_type_encoded"] = df["task_type"].map(task_type_map).fillna(-1).astype(int)

# learner_group: G1 → 1, G2 → 2, G3 → 3, G4 → 4
learner_group_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4}
df["learner_group_encoded"] = df["learner_group"].map(learner_group_map).fillna(0).astype(int)

# hint_viewed: boolean → int
df["hint_viewed"] = df["hint_viewed"].fillna(False).astype(int)

print("Task type distribution:")
print(df["task_type_encoded"].value_counts().rename(index={0: "sql_block", 1: "sql_text"}))
print("\nLearner group distribution:")
print(df["learner_group_encoded"].value_counts().sort_index())

## Step 4 — Create at_risk label

`at_risk = 1` when `COALESCE(review_score, auto_score) < max_score × 0.6`, or no submission exists.

In [ ]:
df["effective_score"] = df["review_score"].combine_first(df["auto_score"])
df["pass_threshold"]  = df["max_score"] * PASS_THRESHOLD_RATIO
df["at_risk"] = (
    df["effective_score"].isna() |
    (df["effective_score"] < df["pass_threshold"])
).astype(int)

balance = df["at_risk"].value_counts()
print("Label distribution:")
print(balance)
print(f"\nat_risk rate : {df['at_risk'].mean()*100:.1f}%")

In [ ]:
LEAKAGE_COLS = [
    "review_score", "auto_score", "final_score", "effective_score",
    "is_passed", "pass_threshold", "max_score",
    "auth_user_id", "email", "display_name", "full_name",
]

ORACLE_COLS = [
    "c1_correctness_result",
    "c2_semantic_consistency",
    "l1_logical_reasoning",
    "l2_learning_process",
    "l3_difficulty_complexity",
]

ID_COLS   = ["academy_member_id", "task_id", "batch_code"]
LABEL_COL = "at_risk"

BASELINE_FEATURE_COLS = [
    # runs / attempts
    "total_run_count",
    "total_attempt_count",
    "attempt_count",
    # timing
    "time_to_first_correct_sec",
    "session_duration_sec",
    "avg_execution_time_ms",
    # errors / attempt quality
    "first_attempt_correct",
    "attempt_correct_ratio",
    "error_type_diversity",
    # session behavior
    "hint_viewed",
    # task metadata
    "task_difficulty_level",
    "task_type_encoded",
    # learner group
    "learner_group_encoded",
    # Phase 5 M5.3: block event features (0 for non-sql_block sessions)
    *BLOCK_FEATURE_COLS,
]

# Verify all expected feature columns exist
missing = [c for c in BASELINE_FEATURE_COLS if c not in df.columns]
if missing:
    print(f"[WARN] Missing feature columns: {missing}")
else:
    print(f"[OK] All {len(BASELINE_FEATURE_COLS)} baseline feature columns present "
          f"({len(BASELINE_FEATURE_COLS) - len(BLOCK_FEATURE_COLS)} core + {len(BLOCK_FEATURE_COLS)} block)")

# Confirm leakage columns are excluded
leaked = [c for c in LEAKAGE_COLS + ORACLE_COLS if c in BASELINE_FEATURE_COLS]
assert not leaked, f"Leakage detected in feature list: {leaked}"
print("[OK] No leakage columns in baseline feature list")

In [ ]:
df_model = df.dropna(subset=BASELINE_FEATURE_COLS).copy()
print(f"Rows after dropping nulls in feature columns: {len(df_model):,}")

X = df_model[BASELINE_FEATURE_COLS].astype(float)
y = df_model[LABEL_COL].astype(int)
groups = df_model["academy_member_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test  = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test  = y.iloc[test_idx].reset_index(drop=True)

train_students = set(groups.iloc[train_idx])
test_students  = set(groups.iloc[test_idx])
overlap = train_students & test_students
assert not overlap, f"Student overlap detected: {overlap}"

print(f"\nTrain / Test split:")
print(f"  Train rows : {len(X_train):,}  ({len(train_students)} students)")
print(f"  Test rows  : {len(X_test):,}  ({len(test_students)} students)")
print(f"  at_risk rate (train) : {y_train.mean()*100:.1f}%")
print(f"  at_risk rate (test)  : {y_test.mean()*100:.1f}%")
print(f"  Student overlap      : {len(overlap)} (must be 0) ✅")

## Step 6 — Group split by learner identity

Using `GroupShuffleSplit` with `groups = academy_member_id`.
Same learner must not appear in both train and test.

In [ ]:
X_train.to_parquet(PROCESSED_DIR / "X_baseline_train.parquet", index=False)
X_test.to_parquet(PROCESSED_DIR  / "X_baseline_test.parquet",  index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False, header=True)
y_test.to_csv(PROCESSED_DIR  / "y_test.csv",  index=False, header=True)

split_metadata = {
    "snapshot_date":            SNAPSHOT_DATE,
    "batch_code":               BATCH_CODE,
    "pass_threshold_ratio":     PASS_THRESHOLD_RATIO,
    "test_size":                TEST_SIZE,
    "random_state":             RANDOM_STATE,
    "split_method":             "GroupShuffleSplit",
    "group_key":                "academy_member_id",
    "total_rows":               len(df_model),
    "train_rows":               len(X_train),
    "test_rows":                len(X_test),
    "train_students":           len(train_students),
    "test_students":            len(test_students),
    "at_risk_rate_train":       round(float(y_train.mean()), 4),
    "at_risk_rate_test":        round(float(y_test.mean()),  4),
    "baseline_features":        BASELINE_FEATURE_COLS,
    "block_features":           BLOCK_FEATURE_COLS,
    "n_block_features":         len(BLOCK_FEATURE_COLS),
    "block_event_csv_loaded":   df_event is not None,
    "oracle_features":          ORACLE_COLS,
    "leakage_excluded":         LEAKAGE_COLS,
}

with open(PROCESSED_DIR / "split_metadata.json", "w") as f:
    json.dump(split_metadata, f, indent=2)

print("Saved:")
for p in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {p}  ({p.stat().st_size:,} bytes)")

## Save outputs